# Geophysical Waveform Inversion: Kaggle Pipeline

Run the cells from top to bottom. The notebook downloads the GitHub repository when needed, validates the environment and data, computes statistics, performs a one-epoch preflight, and provides separate cells for formal training, inference, and submission validation.

In [1]:
# Detect the execution environment and reset stale Kaggle output state.
# This must be the first code cell: it wipes leftover artifacts from previous
# runs so every training session starts from a clean slate.
import os
import shutil
from pathlib import Path

IS_KAGGLE = Path("/kaggle").is_dir()
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'local'}")

if IS_KAGGLE:
    # Remove every file/folder under /kaggle/output. Add more directories here
    # (e.g. "/kaggle/working") if you also want to wipe past run outputs.
    CLEAN_TARGETS = ["/kaggle/working"]
    for target in CLEAN_TARGETS:
        target_path = Path(target)
        if not target_path.is_dir():
            print(f"[clean] {target} does not exist; skipping")
            continue
        removed = 0
        for child in target_path.iterdir():
            if child.is_dir() and not child.is_symlink():
                shutil.rmtree(child)
            else:
                child.unlink()
            removed += 1
        print(f"[clean] removed {removed} item(s) from {target}")
else:
    print("[clean] local environment; no Kaggle cleanup performed")


Environment: Kaggle
[clean] removed 1 item(s) from /kaggle/working


In [2]:
# Kaggle setup: use the current repository or clone it from GitHub.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/fancyleo/Geophysical-Waveform-Inversion.git"
CURRENT_DIR = Path.cwd()
REPO_DIR = next(
    (path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
     if (path / "working_space" / "train.py").exists()),
    CURRENT_DIR / "Geophysical-Waveform-Inversion",
)
if not (REPO_DIR / "working_space" / "train.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif (REPO_DIR / ".git").is_dir():
    # Refresh the checkout so this notebook never runs stale code from an
    # earlier commit. Local changes in working_space/ must be pushed first.
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False
    )

# Kaggle mounts competition data separately from the code repository.
KAGGLE_DATA_ROOT = Path("/kaggle/input/competitions/waveform-inversion")
if not (KAGGLE_DATA_ROOT / "train_samples").is_dir():
    raise FileNotFoundError(f"Training data not found: {KAGGLE_DATA_ROOT / 'train_samples'}")
if not (KAGGLE_DATA_ROOT / "test").is_dir():
    raise FileNotFoundError(f"Test data not found: {KAGGLE_DATA_ROOT / 'test'}")

os.environ["WAVEFORM_DATA_ROOT"] = str(KAGGLE_DATA_ROOT)
os.environ["WAVEFORM_OUTPUT_ROOT"] = "/kaggle/working"
# Reduce CUDA allocator fragmentation during large activation allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Disable tqdm bars in training subprocesses to avoid flooding the cell output buffer.
os.environ["TQDM_DISABLE"] = "1"
# Limit glibc malloc arenas so freed memory is returned to the OS instead of
# accumulating in process RSS across epochs.
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
sys.path.insert(0, str(REPO_DIR / "working_space"))

# Drop any previously imported project modules so this notebook always uses the
# latest local code instead of a stale in-memory module from an earlier run.
for stale in ("config", "data", "model", "training", "utils", "train", "infer"):
    sys.modules.pop(stale, None)

# --- Subprocess interpreter probe ------------------------------------------
# The kernel itself can import torch, but the interpreter we launch in
# subprocesses must ALSO be able to import torch AND see the GPUs. A CPU-only
# torch would make DDP's torch.cuda.set_device() crash. Probe each candidate
# and report its CUDA status so failures are obvious.
def _torch_probe(python):
    """Return (ok, stdout, stderr) for a torch import probe in ``python``."""
    try:
        r = subprocess.run(
            [python, "-c",
             "import sys, torch; print(sys.executable); "
             "print('torch', torch.__version__); "
             "print('cuda_available', torch.cuda.is_available()); "
             "print('device_count', torch.cuda.device_count())"],
            capture_output=True, text=True, timeout=120,
        )
        return r.returncode == 0, r.stdout.strip(), r.stderr.strip()
    except Exception as exc:
        return False, "", str(exc)

# Prefer the kernel's own interpreter (it already proved it can see the GPUs),
# then fall back to common Kaggle conda locations.
_candidates = [sys.executable, str(Path(sys.prefix) / "bin" / "python"), "/opt/conda/bin/python"]
_seen = set()
PYTHON = None
TORCH_INFO = ""
for _cand in _candidates:
    _cand = os.path.normpath(_cand)
    if _cand in _seen or not Path(_cand).exists():
        continue
    _seen.add(_cand)
    _ok, _out, _err = _torch_probe(_cand)
    if _ok:
        PYTHON = _cand
        TORCH_INFO = _out
        break
    print(f"[probe] {_cand}: torch import FAILED")
    if _err:
        print(_err)
if PYTHON is None:
    raise RuntimeError("No Python with torch found; candidates: " + ", ".join(_candidates))
print(f"[probe] subprocess interpreter OK:\n{TORCH_INFO}")

def run_cmd(cmd, **kwargs):
    """Run a subprocess, stream its output, and surface a useful error."""
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        print("[error] stdout tail:\n" + (result.stdout[-4000:] if result.stdout else ""), end="")
        print("[error] stderr:\n" + (result.stderr or ""), end="")
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}"
        )
    if result.stderr:
        print("[stderr]\n" + result.stderr, end="")
    return result

print(f"Repository: {REPO_DIR}")
print(f"Input root: {os.environ['WAVEFORM_DATA_ROOT']}")
print(f"Output root: {os.environ['WAVEFORM_OUTPUT_ROOT']}")
print(f"CUDA allocator: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"tqdm disabled: {os.environ['TQDM_DISABLE']}")
print(f"MALLOC_ARENA_MAX: {os.environ['MALLOC_ARENA_MAX']}")


Cloning into '/kaggle/working/Geophysical-Waveform-Inversion'...


[probe] subprocess interpreter OK:
/usr/bin/python3
torch 2.10.0+cu128
cuda_available True
device_count 2
Repository: /kaggle/working/Geophysical-Waveform-Inversion
Input root: /kaggle/input/competitions/waveform-inversion
Output root: /kaggle/working
CUDA allocator: expandable_segments:True
tqdm disabled: 1
MALLOC_ARENA_MAX: 2


In [3]:
# Copy the repository's working_space contents into /kaggle/working.
import shutil

WORKING_SPACE_DIR = REPO_DIR / "working_space"
KAGGLE_WORKING_DIR = Path("/kaggle/working")
KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    WORKING_SPACE_DIR,
    KAGGLE_WORKING_DIR,
    dirs_exist_ok=True,
)
print(f"Copied: {WORKING_SPACE_DIR}")
print(f"Destination: {KAGGLE_WORKING_DIR}")

Copied: /kaggle/working/Geophysical-Waveform-Inversion/working_space
Destination: /kaggle/working


In [4]:
import importlib
import torch

for package_name in ["numpy", "matplotlib", "sklearn", "tqdm", "torch"]:
    module = importlib.import_module(package_name)
    print(f"{package_name}: {getattr(module, '__version__', 'available')}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

numpy: 2.0.2
matplotlib: 3.10.0
sklearn: 1.6.1
tqdm: 4.67.3
torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
from config import Cfg, select_families
from train import find_pairs
import os

print(f"Input root: {Cfg.input_root}")
print(f"Training data: {Cfg.train_data_dir}")
print(f"Test data: {Cfg.test_data_dir}")
print(f"Training data exists: {Cfg.train_data_dir.is_dir()}")
print(f"Test data exists: {Cfg.test_data_dir.is_dir()}")

# Diagnostic: show what is actually under the training data root.
train_root = str(Cfg.train_data_dir)
if os.path.isdir(train_root):
    entries = sorted(os.listdir(train_root))
    print(f"[diag] train_samples entries ({len(entries)}): {entries[:20]}")
    for entry in entries[:3]:
        sub = os.path.join(train_root, entry)
        if os.path.isdir(sub):
            print(f"[diag]   {entry}/ -> {sorted(os.listdir(sub))[:10]}")

families = select_families("all")
pairs = find_pairs(train_root, families)
print(f"Families: {len(families)}; paired files: {len(pairs)}")
assert pairs, "No training pairs were found."


Input root: /kaggle/input/competitions/waveform-inversion
Training data: /kaggle/input/competitions/waveform-inversion/train_samples
Test data: /kaggle/input/competitions/waveform-inversion/test
Training data exists: True
Test data exists: True
[diag] train_samples entries (10): ['CurveFault_A', 'CurveFault_B', 'CurveVel_A', 'CurveVel_B', 'FlatFault_A', 'FlatFault_B', 'FlatVel_A', 'FlatVel_B', 'Style_A', 'Style_B']
[diag]   CurveFault_A/ -> ['seis2_1_0.npy', 'seis4_1_0.npy', 'vel2_1_0.npy', 'vel4_1_0.npy']
[diag]   CurveFault_B/ -> ['seis6_1_0.npy', 'seis8_1_0.npy', 'vel6_1_0.npy', 'vel8_1_0.npy']
[diag]   CurveVel_A/ -> ['data', 'model']
[info] total paired files: 20
Families: 10; paired files: 20


In [6]:
run_cmd([PYTHON, str(REPO_DIR / "working_space" / "test_unet.py")])
run_cmd([PYTHON, str(REPO_DIR / "working_space" / "smoke_test.py")])


forward output: (2, 70, 70)
backward pass: ok
[smoke] fake data at: /tmp/wi_smoke_1y7z4x7c
[info] total paired files: 6
[smoke] paired files: 6  (expected 6)
[smoke] vel_mean=3500.7  vel_std=576.8
[smoke] submission lines: 141 (expected 1+2*70=141)
[smoke] first data line (truncated): oid000_y_0,3614.4,4369.1,3507.6 ..., 3508.2
Smoke test passed: data loading and submission format are valid
[smoke] cleaned up /tmp/wi_smoke_1y7z4x7c


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/Geophysical-Waveform-Inversion/working_space/smoke_test.py'], returncode=0, stdout='[smoke] fake data at: /tmp/wi_smoke_1y7z4x7c\n[info] total paired files: 6\n[smoke] paired files: 6  (expected 6)\n[smoke] vel_mean=3500.7  vel_std=576.8\n[smoke] submission lines: 141 (expected 1+2*70=141)\n[smoke] first data line (truncated): oid000_y_0,3614.4,4369.1,3507.6 ..., 3508.2\nSmoke test passed: data loading and submission format are valid\n[smoke] cleaned up /tmp/wi_smoke_1y7z4x7c\n', stderr='')

In [7]:
# Ensure the velocity statistics JSON exists before training so train.py loads
# the true dataset mean/std instead of falling back to Cfg.vel_mean / vel_std
# (which would also emit the "[warn] statistics file not found" message).
from config import Cfg, select_families, stats_path_for_families, load_velocity_stats

families = select_families("all")
stats_path = stats_path_for_families(families)
if stats_path.is_file():
    print(f"[info] using existing velocity statistics: {stats_path}")
else:
    print(f"[info] computing velocity statistics: {stats_path}")
    run_cmd([
        PYTHON, str(REPO_DIR / "working_space" / "compute_stats.py"),
        "--data_dir", str(Cfg.train_data_dir), "--family", "all",
    ])
vel_mean, vel_std = load_velocity_stats(stats_path)
print(f"[info] velocity mean={vel_mean:.2f}  std={vel_std:.2f}")
assert stats_path.is_file(), f"Statistics still missing: {stats_path}"


[info] computing velocity statistics: /kaggle/working/stats/velocity_stats_flatvel_a_flatvel_b_curvevel_a_curvevel_b_flatfault_a_flatfault_b_curvefault_a_curvefault_b_style_a_style_b.json
[info] selected families: FlatVel_A, FlatVel_B, CurveVel_A, CurveVel_B, FlatFault_A, FlatFault_B, CurveFault_A, CurveFault_B, Style_A, Style_B
[info] found 20 velocity .npy files
[stats] count   = 49000000
[stats] mean    = 2916.82
[stats] std     = 817.36
[stats] min     = 1500.00
[stats] max     = 4500.00
[stats] median  = 2898.00
[done] saved velocity statistics to /kaggle/working/stats/velocity_stats_flatvel_a_flatvel_b_curvevel_a_curvevel_b_flatfault_a_flatfault_b_curvefault_a_curvefault_b_style_a_style_b.json
[info] velocity mean=2916.82  std=817.36


In [8]:
# Run a multi-GPU DDP smoke test before the formal training job.
# train.py --test_run uses the flat families, 3 epochs, and memory monitoring;
# --parallel_mode data_parallel with --nproc_per_node 2 launches DDP over the
# available T4 GPUs. Watch the [mem] delta: it should stay flat across epochs,
# otherwise a host memory leak is present and you should fall back to single.
TRAIN_DIR = Cfg.train_data_dir
OUTPUT_DIR = Cfg.output_dir
NPROC = 2 if torch.cuda.device_count() > 1 else 1
run_cmd([
    PYTHON, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--test_run",
    "--batch_size", "4",
    "--num_workers", "0",
    "--parallel_mode", "data_parallel",
    "--nproc_per_node", str(NPROC),
    "--log_memory",
])
print(f"[info] DDP smoke test finished with nproc={NPROC}")


[info] test_run enabled: flat families, 3 epochs, memory monitoring on
[warn] statistics file not found: /kaggle/working/stats/velocity_stats_flatvel_a_flatvel_b_flatfault_a_flatfault_b.json; using Cfg.vel_mean and Cfg.vel_std
[info] total paired files: 8
[warn] statistics file not found: /kaggle/working/stats/velocity_stats_flatvel_a_flatvel_b_flatfault_a_flatfault_b.json; using Cfg.vel_mean and Cfg.vel_std
[info] selected families: FlatVel_A, FlatVel_B, FlatFault_A, FlatFault_B
[info] total paired files: 8
[info] train samples: 3500, val samples: 500
[info] using DistributedDataParallel, world_size=2
[info] model params: 24.29M
[mem] epoch 001  host_rss=4455 MiB  delta=+3303 MiB
[mem] epoch 001  cuda_allocated=463 MiB  cuda_reserved=1436 MiB
epoch 001  train_mae_norm=0.6319  val_mae_norm=0.4339  val_mae_raw=354.68
  saved best (val_mae_raw=354.68)
[mem] epoch 002  host_rss=5636 MiB  delta=+4484 MiB
[mem] epoch 002  cuda_allocated=463 MiB  cuda_reserved=1456 MiB
epoch 002  train_mae_n

In [ ]:
# Run this cell for the formal training job.
# Recommendation: keep single mode.
# - DDP works (smoke test passed) but showed host_rss growth (~+0.5-1.2 GB per
#   epoch, 4455->6192 MiB over 3 epochs), which is risky over 60 epochs.
# - Timing from your T4 single-GPU measurement (30 epochs x batch 4 = 4h):
#   batch 8 halves the steps per epoch, so 60 epochs x batch 8 is ~7-8h, within
#   the 12h Kaggle limit.
# - Watch the [mem] delta across epochs: it should stay flat in single mode.
EPOCHS = 60
BATCH_SIZE = 8
FORMAL_PARALLEL_MODE = "single"   # keep single; DDP host_rss grows over 60 epochs
FORMAL_NPROC = 2 if FORMAL_PARALLEL_MODE in ("data_parallel", "ddp") and torch.cuda.device_count() > 1 else 1
run_cmd([
    PYTHON, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--family", "all", "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", "0",
    "--parallel_mode", FORMAL_PARALLEL_MODE,
    "--nproc_per_node", str(FORMAL_NPROC),
    "--log_memory",
])


In [ ]:
run_dirs = sorted(OUTPUT_DIR.glob("**/model_*/"), key=lambda path: path.stat().st_mtime)
assert run_dirs, "No training run directory was found."
latest_run = run_dirs[-1]
checkpoint = latest_run / "best_unet.pth"
assert checkpoint.exists(), f"Checkpoint not found: {checkpoint}"
submission_path = OUTPUT_DIR / "submission.csv"
print(f"Using checkpoint: {checkpoint}")
run_cmd([
    PYTHON, str(REPO_DIR / "working_space" / "infer.py"),
    "--ckpt", str(checkpoint), "--test_dir", str(Cfg.test_data_dir),
    "--out", str(submission_path), "--batch_size", str(Cfg.infer_batch_size),
])


In [ ]:
import pandas as pd
submission = pd.read_csv(submission_path)
expected_columns = 1 + len(range(Cfg.submission_x_start, Cfg.submission_x_stop, Cfg.submission_x_step))
print(f"Submission shape: {submission.shape}")
print(f"Missing values: {int(submission.isna().sum().sum())}")
assert submission.shape[1] == expected_columns
assert submission.isna().sum().sum() == 0
print("Submission validation passed.")